# 扩展数据集 vs 原有数据集 — STFT 心音去噪

一个问题：**把 CirCor 加进训练集，模型在自己的设备域上会不会变好。**

两条臂除了训练用的干净心音来源，其他一切相同 —— 模型、超参、噪声、验证集、seed：

| | 训练干净心音 | 验证集 |
|---|---|---|
| 臂 A 原有 | device 3936 窗 | device 留出受试者 |
| 臂 B 扩展 | device + CirCor | **同一个** device 留出受试者 |

验证集两臂共用且同 seed，所以比较是配对的：报告逐样本差值和置信区间。


## 0. 环境


In [ ]:
!nvidia-smi -L || echo 'no GPU — Runtime > Change runtime type > GPU'


把本地 `stft/` 整个文件夹放到 Drive（例如 `MyDrive/EE596/stft`），下面复制到本地磁盘再跑，
训练时的小文件 IO 就不用走 Drive。


In [ ]:
import os, shutil, sys, json
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')

DRIVE_STFT = Path('/content/drive/MyDrive/EE596/stft')   # <-- 改这里
WORK = Path('/content/stft')

assert DRIVE_STFT.is_dir(), f'{DRIVE_STFT} 不存在：把 stft/ 上传到 Drive，或改上面的路径'
if WORK.exists():
    shutil.rmtree(WORK)
shutil.copytree(DRIVE_STFT, WORK, ignore=shutil.ignore_patterns('__pycache__', '.git', 'checkpoints', 'outputs'))
os.chdir(WORK)
sys.path.insert(0, str(WORK))
os.environ['PYTHONPATH'] = str(WORK)

!pip install -q soundfile
import torch
print(WORK, '| torch', torch.__version__, '| cuda', torch.cuda.is_available())


## 1. CirCor pool（只有臂 B 需要，建一次）

449 MB 下载 + 采样，十几分钟。建好后存回 Drive，下次 session 直接复用。

采样只在 TSV 非零 state 的连续标注段内取窗，并丢掉 `Murmur=Unknown` 的受试者 ——
state 0 是 CirCor 自带的信号质量标签，那些段里的噪声源官方列了听诊器摩擦、说话、小孩哭笑。
把它们当干净目标喂给去噪模型，等于教它输出噪声。


In [ ]:
POOL = Path('data/circor/circor_pool_4khz_2s.npz')
DRIVE_POOL = DRIVE_STFT / 'data' / 'circor' / POOL.name
POOL.parent.mkdir(parents=True, exist_ok=True)

if DRIVE_POOL.is_file() and not POOL.is_file():
    shutil.copy(DRIVE_POOL, POOL)
    print('从 Drive 复用:', POOL)
print('pool 存在:', POOL.is_file())


In [ ]:
if not POOL.is_file():
    !wget -q --show-progress -O /content/circor.zip https://physionet.org/content/circor-heart-sound/get-zip/1.0.3/
    !unzip -q -o /content/circor.zip -d /content
    !ls -d /content/circor-heart-sound-*


In [ ]:
if not POOL.is_file():
    !PYTHONPATH=. python scripts/build_circor_pool.py \
        --root /content/circor-heart-sound-1.0.3 --out {POOL} --windows-per-patient 12
    DRIVE_POOL.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy(POOL, DRIVE_POOL)
    print('cached ->', DRIVE_POOL)


## 2. 体检两条臂

确认格式、纯净度、归一化前的尺度差、train/val 没有受试者泄漏。看两个数字：
`largest device/CirCor level gap`（为什么必须归一）和 CirCor 的 bpm 中位数（device 是 73）。


In [ ]:
!PYTHONPATH=. python scripts/audit_pools.py --no_circor 2>&1 | tail -22


In [ ]:
!PYTHONPATH=. python scripts/audit_pools.py --json outputs/pool_audit.json 2>&1 | tail -30


## 3. 单测 + smoke（确认整条路走得通）


In [ ]:
!PYTHONPATH=. python scripts/check_frequency_pipeline.py


In [ ]:
!PYTHONPATH=. python train_frequency.py --smoke --device cpu 2>&1 | tail -12


## 4. 两条臂

两条命令**只差 `--no_circor` 和输出目录**。epochs / batch / seed 必须一致，否则比的就不是数据集了。
改 `SEED` 重跑一遍，可以看差值是否扛得住训练方差 —— 这才是这个实验真正的终点。


In [ ]:
EPOCHS = 60
BATCH  = 16
SEED   = 2026
TAG    = f'seed{SEED}'
print(f'臂 A -> checkpoints/{TAG}/device_only,  臂 B -> checkpoints/{TAG}/combined')


In [ ]:
# 臂 A：原有 device 数据集
!PYTHONPATH=. python train_frequency.py --no_circor \
    --epochs {EPOCHS} --batch_size {BATCH} --seed {SEED} \
    --output_dir checkpoints/{TAG}/device_only


In [ ]:
# 臂 B：扩展数据集（device + CirCor）
!PYTHONPATH=. python train_frequency.py \
    --epochs {EPOCHS} --batch_size {BATCH} --seed {SEED} \
    --output_dir checkpoints/{TAG}/combined


## 5. 训练曲线（同一张图上两条臂）


In [ ]:
import matplotlib.pyplot as plt

def history(run):
    path = Path('checkpoints') / TAG / run / 'history.jsonl'
    return [json.loads(line) for line in path.read_text().splitlines() if line.strip()]

runs = {'device_only': history('device_only'), 'combined': history('combined')}
figure, axes = plt.subplots(1, 3, figsize=(16, 4))
for label, records in runs.items():
    epochs = [r['epoch'] + 1 for r in records]
    axes[0].plot(epochs, [r['train']['loss_mean'] for r in records], label=f'{label} train')
    axes[0].plot(epochs, [r['validation']['loss_mean'] for r in records], '--', label=f'{label} val')
    axes[1].plot(epochs, [r['validation']['si_sdr_improvement_db_mean'] for r in records], label=label)
    axes[2].plot(epochs, [r['validation']['snr_improvement_db_mean'] for r in records], label=label)
axes[0].set_title('loss'); axes[1].set_title('val SI-SDRi (dB)'); axes[2].set_title('val SNRi (dB)')
for axis in axes[1:]:
    axis.axhline(0, color='k', linewidth=0.6)
for axis in axes:
    axis.set_xlabel('epoch'); axis.legend(fontsize=8)
figure.tight_layout()

for label, records in runs.items():
    best = max(records, key=lambda r: 0.5 * (r['validation']['si_sdr_improvement_db_mean'] + r['validation']['snr_improvement_db_mean']))
    print(f"{label:12} best epoch {best['epoch'] + 1:3d}  "
          f"SI-SDRi {best['validation']['si_sdr_improvement_db_mean']:+.2f} dB  "
          f"SNRi {best['validation']['snr_improvement_db_mean']:+.2f} dB  "
          f"corr {best['validation']['correlation_mean']:.3f}")


## 6. 配对比较 ← 结论在这里

评测集在脚本里现场构建，不读任何训练目录：device 留出受试者的干净心音配 device val 噪声，
七个固定 SNR 档各 200 样本，固定 seed。两个 checkpoint 吃到逐字节相同的混合，所以可以做配对检验。


In [ ]:
!PYTHONPATH=. python scripts/compare_datasets.py \
    --device_only checkpoints/{TAG}/device_only/best.pt \
    --combined    checkpoints/{TAG}/combined/best.pt \
    --output_dir  outputs/comparison_{TAG}


In [ ]:
import IPython.display as display
display.display(display.Image(f'outputs/comparison_{TAG}/comparison.png'))


**怎么读**：右边那张图是 `combined − device_only` 的配对差值和 95% 区间。
区间跨过 0 的档位，就是这一档分不出差别。

而且整体上：这只是两个 checkpoint 在一个评测集上的差异，不含训练种子方差。
差值量级和区间宽度可比 → 还不算证据（不管正负）。换 SEED 重跑第 4 节再看。


## 7. 同步回 Drive


In [ ]:
DEST = DRIVE_STFT.parent / 'stft_runs'
DEST.mkdir(parents=True, exist_ok=True)
for folder in ('checkpoints', 'outputs'):
    source = Path(folder)
    if source.exists():
        target = DEST / folder
        if target.exists():
            shutil.rmtree(target)
        shutil.copytree(source, target)
        print('copied', source, '->', target)
